<!-- paper-paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 구현 이유와 텐서 shape를 설명해 보세요.

# 07. LSTM Sequence to Sequence — Sequence to Sequence Learning with Neural Networks

- **원 논문:** [Sequence to Sequence Learning with Neural Networks](https://arxiv.org/abs/1409.3215)
- **저자 / 발표:** Ilya Sutskever, Oriol Vinyals, Quoc V. Le · NeurIPS 2014 (2014)
- **난이도 / 예상 시간:** 중급 · 약 75분
- **선수 지식:** LSTM, teacher forcing, cross entropy, autoregressive decoding

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

LSTM encoder의 마지막 hidden/cell state를 LSTM decoder의 초기 상태로 넘기고, BOS 입력과 한 칸 이동한 target으로 조건부 다음-token 확률을 학습한다. 논문의 source 순서 뒤집기 기법과 greedy autoregressive decoding까지 작은 복사 과제로 확인한다.

**원 논문과 다른 것**

원 논문은 WMT'14 약 1,200만 문장 쌍, 160k/80k source/target vocabulary와 4층 1000-unit LSTM(약 3.8억 파라미터)을 여러 GPU에서 학습했다. 여기서는 8개 길이-3 수열을 한 층 LSTM으로 암기해 정보 흐름만 재현한다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §2, 식 (1): 조건부 확률을 token별 확률의 곱으로 분해 | `decoder_inputs`/`next_token_targets` shift와 teacher-forced NLL | decoder input과 target이 정확히 한 칸 어긋나는지 assert |
| §2: 첫 LSTM의 마지막 state가 고정 길이 표현 v를 구성 | `Seq2Seq.forward`가 encoder `(h, c)`를 decoder 초기 상태로 전달 | logit shape와 encoder/decoder state shape 검사 |
| §2: 두 번째 LSTM이 v에 조건부로 출력 문장을 생성 | `greedy_decode`가 BOS부터 예측 token을 다시 입력 | token accuracy와 exact-sequence accuracy를 모두 출력 |
| §3.2: source sentence의 단어 순서를 뒤집으면 minimal time lag 감소 | `build_seq2seq_batch`의 `encoder_ids = flip(source_ids)` | 첫 행이 정확히 역순인지 assert |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
p(y_{1:T}\mid x_{1:S})=\prod_{t=1}^{T}
p(y_t\mid y_{<t},c),\qquad c=\operatorname{Encoder}(x_{1:S})
$$

- $c$는 source sequence를 요약한 encoder state, $y_{<t}$는 이전 target token입니다.
- decoder는 teacher forcing으로 학습하고 autoregressive loop로 추론합니다.
- vocabulary, encoder, decoder, sequence loss, greedy decode를 명시적으로 분리합니다.
- token $[B,S]$는 recurrent state $[L,B,H]$를 거쳐 target logit $[B,T,V]$가 됩니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §2, 식 (1): 조건부 확률을 token별 확률의 곱으로 분해
- **노트북 구현:** `decoder_inputs`/`next_token_targets` shift와 teacher-forced NLL
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** decoder input과 target이 정확히 한 칸 어긋나는지 assert를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 원 논문은 WMT'14 약 1,200만 문장 쌍, 160k/80k source/target vocabulary와 4층 1000-unit LSTM(약 3.8억 파라미터)을 여러 GPU에서 학습했다. 여기서는 8개 길이-3 수열을 한 층 LSTM으로 암기해 정보 흐름만 재현한다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 핵심 아이디어

논문의 기본 모형은 한 LSTM이 source를 읽고 마지막 상태로 표현 $v$를 만든 뒤,
다른 LSTM이
$p(T'\mid S)=\prod_t p(y_t\mid v,y_1,\ldots,y_{t-1})$를 모델링한다.
학습 중에는 실제 이전 token을 넣지만(teacher forcing), 평가 때는 자신의 예측을
다시 입력해야 한다. 논문 §3.2처럼 encoder 입력만 뒤집고 정답 순서는 유지한다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** source [B,S] → recurrent state [L,B,H] → target logits [B,T,V]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

SEED = 707
torch.manual_seed(SEED)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
print(ACCELERATOR.summary())
PAD, BOS, EOS = 0, 1, 2
VOCAB_SIZE = 8

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2, 식 (1): 조건부 확률을 token별 확률의 곱으로 분해 / §3.2: source sentence의 단어 순서를 뒤집으면 minimal time lag 감소
- **이 셀의 책임:** `decoder_inputs`/`next_token_targets` shift와 teacher-forced NLL / `build_seq2seq_batch`의 `encoder_ids = flip(source_ids)`
- **Tensor shape 계약:** source [B,S] → recurrent state [L,B,H] → target logits [B,T,V]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** decoder input과 target이 정확히 한 칸 어긋나는지 assert / 첫 행이 정확히 역순인지 assert. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
source_ids = torch.tensor(
    [
        [3, 4, 5],
        [4, 5, 6],
        [5, 6, 7],
        [7, 6, 5],
        [3, 6, 4],
        [4, 7, 3],
        [5, 3, 7],
        [6, 4, 3],
    ]
)


def build_seq2seq_batch(source):
    if source.ndim != 2:
        raise ValueError("source must have shape [batch, length]")
    encoder_input = torch.flip(source, dims=(1,))
    batch = source.shape[0]
    bos_column = torch.full((batch, 1), BOS, dtype=torch.long)
    eos_column = torch.full((batch, 1), EOS, dtype=torch.long)
    decoder_target = torch.cat(
        (bos_column, source, eos_column),
        dim=1,
    )
    return encoder_input, decoder_target


encoder_ids, target_ids = build_seq2seq_batch(source_ids)
source_ids, encoder_ids, target_ids = ACCELERATOR.move(
    source_ids, encoder_ids, target_ids
)

assert torch.equal(encoder_ids[0], encoder_ids.new_tensor([5, 4, 3]))
assert torch.all(target_ids[:, 0] == BOS) and torch.all(target_ids[:, -1] == EOS)
print("encoder:", encoder_ids[0].tolist(), "target:", target_ids[0].tolist())

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §2: 첫 LSTM의 마지막 state가 고정 길이 표현 v를 구성
- **이 셀의 책임:** `Seq2Seq.forward`가 encoder `(h, c)`를 decoder 초기 상태로 전달
- **Tensor shape 계약:** source [B,S] → recurrent state [L,B,H] → target logits [B,T,V]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** logit shape와 encoder/decoder state shape 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, vocab_size, embedding_dim=16, hidden_dim=32):
        super().__init__()
        self.source_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.target_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.encoder = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.decoder = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def encode(self, source):
        _, state = self.encoder(self.source_embedding(source))
        return state

    def decode_teacher_forced(self, decoder_inputs, state):
        decoded, next_state = self.decoder(self.target_embedding(decoder_inputs), state)
        return self.output(decoded), next_state

    def forward(self, source, decoder_inputs):
        state = self.encode(source)
        logits, _ = self.decode_teacher_forced(decoder_inputs, state)
        return logits


torch.manual_seed(SEED)
model = Seq2Seq(VOCAB_SIZE).to(DEVICE)
probe_logits = model(encoder_ids, target_ids[:, :-1])
assert probe_logits.shape == (len(source_ids), target_ids.shape[1] - 1, VOCAB_SIZE)

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** §2, 식 (1): 조건부 확률을 token별 확률의 곱으로 분해
- **이 셀의 책임:** `decoder_inputs`/`next_token_targets` shift와 teacher-forced NLL
- **Tensor shape 계약:** source [B,S] → recurrent state [L,B,H] → target logits [B,T,V]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** decoder input과 target이 정확히 한 칸 어긋나는지 assert. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
decoder_inputs = target_ids[:, :-1]
next_token_targets = target_ids[:, 1:]
assert torch.equal(decoder_inputs[:, 1:], next_token_targets[:, :-1])

optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
loss_history = []


def train_seq2seq_step(model, source, inputs, targets, optimizer):
    model.train()
    optimizer.zero_grad()
    logits = model(source, inputs)
    loss = F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),
        targets.reshape(-1),
    )
    loss.backward()
    gradient_norm = nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=5.0,
    )
    optimizer.step()
    return float(loss.detach()), float(gradient_norm)


gradient_norms = []
for _ in range(180):
    step_loss, gradient_norm = train_seq2seq_step(
        model,
        encoder_ids,
        decoder_inputs,
        next_token_targets,
        optimizer,
    )
    loss_history.append(step_loss)
    gradient_norms.append(gradient_norm)

assert loss_history[-1] < 0.15 * loss_history[0]
assert torch.isfinite(torch.tensor(gradient_norms)).all()
print(f"teacher-forced loss: {loss_history[0]:.3f} -> {loss_history[-1]:.3f}")

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §2: 두 번째 LSTM이 v에 조건부로 출력 문장을 생성
- **이 셀의 책임:** `greedy_decode`가 BOS부터 예측 token을 다시 입력
- **Tensor shape 계약:** source [B,S] → recurrent state [L,B,H] → target logits [B,T,V]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** token accuracy와 exact-sequence accuracy를 모두 출력. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
@torch.inference_mode()
def greedy_decode(model, source, steps):
    model.eval()
    state = model.encode(source)
    current = torch.full(
        (len(source), 1),
        BOS,
        dtype=torch.long,
        device=source.device,
    )
    predictions = []
    for _ in range(steps):
        logits, state = model.decode_teacher_forced(current, state)
        current = logits[:, -1:].argmax(dim=-1)
        predictions.append(current)
    return torch.cat(predictions, dim=1)


predicted_ids = greedy_decode(model, encoder_ids, target_ids.shape[1] - 1)
expected_ids = target_ids[:, 1:]
token_accuracy = (predicted_ids == expected_ids).float().mean().item()
exact_accuracy = (predicted_ids == expected_ids).all(dim=1).float().mean().item()
assert token_accuracy >= 0.95 and exact_accuracy >= 0.75
print(
    f"token_accuracy={token_accuracy:.1%}, exact_sequence_accuracy={exact_accuracy:.1%}"
)
print(
    "prediction:",
    predicted_ids[0].tolist(),
    "expected:",
    expected_ids[0].tolist(),
)

### 구현 단계 6 · 평가·완료 증거

- **원문 위치:** §2: 두 번째 LSTM이 v에 조건부로 출력 문장을 생성
- **이 셀의 책임:** `greedy_decode`가 BOS부터 예측 token을 다시 입력
- **Tensor shape 계약:** source [B,S] → recurrent state [L,B,H] → target logits [B,T,V]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** token accuracy와 exact-sequence accuracy를 모두 출력. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
per_sequence = (predicted_ids == expected_ids).all(dim=1).float()
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].plot(loss_history)
axes[0].set(title="teacher-forced NLL", xlabel="step", ylabel="loss")
axes[1].bar(range(len(per_sequence)), per_sequence.tolist())
axes[1].set(title="autoregressive exact match", xlabel="sequence", ylim=(0, 1.05))
plt.tight_layout()
plt.show()

## 해석 체크

teacher-forced loss가 작아도 autoregressive exact match가 낮을 수 있는 이유와,
source를 뒤집는 것이 장거리 의존성의 *시간 간격*을 어떻게 줄이는지 적어 보세요.

## 마무리 해설과 다음 실험

1. 핵심 연산과 구현 위치를 원 논문의 용어로 다시 연결해 설명하세요.
2. 이 미니 실험은 `원 논문은 WMT'14 약 1,200만 문장 쌍, 160k/80k source/target vocabulary와 4층 1000-unit LSTM(약 3.8억 파라미터)을 여러 GPU에서 학습했다. 여기서는 8개 길이-3 수열을 한 층 LSTM으로 암기해 정보 흐름만 재현한다.`를 재현하지 않으므로 논문의 최종 성능을 검증한 것이 아닙니다.
3. 다음에는 데이터 크기, 모델 폭, 학습 횟수 중 **하나만** 바꾸고 같은 seed에서 metric을 비교하세요.

정답의 수치 자체보다, shape·확률 정규화·mask·gradient 같은 불변식을 먼저 확인하는 습관이
논문 구현에서 가장 강한 디버깅 도구입니다.